# Topic 8 — Conversation Memory: The SupportAgentMemory Class

Capstone: everything from topics 1-7, wrapped into one clean class a real support agent could actually call.

In [1]:
import redis
from google.cloud.firestore_v1 import ArrayUnion
from setup import firestore_client, REDIS_HOST, REDIS_PORT

class SupportAgentMemory:
    def __init__(self, session_id: str, customer_id: str, redis_host=REDIS_HOST, redis_port=REDIS_PORT):
        self.session_id = session_id
        self.customer_id = customer_id
        self.redis = redis.Redis(host=redis_host, port=redis_port, decode_responses=True)
        self.profile_doc = firestore_client.collection("support_customer_profiles").document(customer_id)
        self.session_key = f"support_session:{session_id}:turns"

    def remember_turn(self, role: str, content: str):
        self.redis.rpush(self.session_key, f"{role}: {content}")
        self.redis.expire(self.session_key, 1800)

    def remember_fact(self, fact: str):
        self.profile_doc.set({"known_issues": ArrayUnion([fact])}, merge=True)

    def recall(self) -> str:
        turns = self.redis.lrange(self.session_key, 0, -1)
        profile = self.profile_doc.get().to_dict() or {}
        return (
            "Current chat:\n" + "\n".join(turns) +
            f"\n\nKnown issues: {profile.get('known_issues', [])}"
        )

### Use it the way a real agent would

In [2]:
memory = SupportAgentMemory(session_id="sess_9001", customer_id="cust_042")

memory.remember_turn("customer", "My app keeps crashing when I open settings.")
memory.remember_fact("App crashes on opening settings - reported today")

print(memory.recall())

Current chat:
customer: My app keeps crashing when I open settings.
customer: My app keeps crashing when I open settings.

Known issues: ['Billing issue resolved Jan 5', 'App crashes on opening settings - reported today']


### This is what would get passed into SupportBot's next Gemini call

In [3]:
context = memory.recall()
print("\n--- This context feeds directly into the model's prompt ---\n")
print(context)


--- This context feeds directly into the model's prompt ---

Current chat:
customer: My app keeps crashing when I open settings.
customer: My app keeps crashing when I open settings.

Known issues: ['Billing issue resolved Jan 5', 'App crashes on opening settings - reported today']


### Now actually give it to the model, and prove it remembers

Until now this notebook only *printed* the context. Here it goes into a real Gemini call. First the same question with **no memory**, then a **brand-new chat session** (`sess_9002`) for the same customer. Redis has nothing for the new session, so anything it knows about the crash has to come from the Firestore profile.

In [4]:
from setup import genai_client, MODEL_FLASH

question = "Hi, is the crash I reported earlier still being looked into?"

without_memory = genai_client.models.generate_content(
    model=MODEL_FLASH,
    contents=f"You are SupportBot, a customer support agent. The customer says: {question}",
)
print("WITHOUT memory:\n" + without_memory.text)

WITHOUT memory:
Hi there! Thanks for reaching out.

Yes, I can definitely check on the status of that crash report for you.

To help me locate it quickly, could you please provide me with one of the following:

*   **The reference number** (ticket ID, case number, etc.) you received when you reported it.
*   **The email address or username** associated with the report.
*   **Roughly when you reported it** and a brief description of the crash, if you don't have a reference number.

Once I have that, I'll look it up and let you know the latest update.


In [5]:
def support_reply(memory: SupportAgentMemory, customer_message: str) -> str:
    memory.remember_turn("customer", customer_message)
    prompt = (
        "You are SupportBot, a customer support agent. Reply directly to the customer's latest "
        "message, in the first person, using ONLY the facts below. Never mention 'memory' or "
        "'context'. If the facts don't cover something, say you don't have that information yet.\n\n"
        f"{memory.recall()}"
    )
    reply = genai_client.models.generate_content(model=MODEL_FLASH, contents=prompt).text
    memory.remember_turn("agent", reply)
    return reply

new_chat = SupportAgentMemory(session_id="sess_9002", customer_id="cust_042")
print("WITH memory:\n" + support_reply(new_chat, question))

WITH memory:
The 'App crashes on opening settings - reported today' is a known issue. I do not have information on whether it is currently being looked into.


**What to notice:** the first answer has to guess, or asks what crash you mean. The second names the settings crash, and it got that from the Firestore profile written earlier, not from anything said in this chat. That is the point of the whole module: the agent's memory outlives the conversation.